# Heterogeneous Treatment Effects, Transportability, and Governance Lab


## Purpose

This lab frames heterogeneous effects and causal governance documentation.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 5000

data = pd.DataFrame({
    "unit_id": [f"unit_{i:05d}" for i in range(1, n + 1)],
    "prior_activity": rng.normal(0, 1, size=n),
    "domain_expertise": rng.normal(0, 1, size=n),
})

data["true_tau"] = (
    0.08
    + 0.04 * (data["prior_activity"] > 0)
    + 0.03 * (data["domain_expertise"] > 0)
)

data["y0"] = (
    0.30
    + 0.08 * data["prior_activity"]
    + 0.04 * data["domain_expertise"]
    + rng.normal(0, 0.05, size=n)
)

data["y1"] = data["y0"] + data["true_tau"]

data.head()

In [ ]:
data["segment"] = np.where(
    (data["prior_activity"] > 0) & (data["domain_expertise"] > 0),
    "high_activity_high_expertise",
    np.where(
        data["prior_activity"] > 0,
        "high_activity",
        "lower_activity"
    )
)

segment_effects = data.groupby("segment").agg(
    units=("unit_id", "count"),
    mean_true_tau=("true_tau", "mean"),
    mean_prior_activity=("prior_activity", "mean"),
    mean_domain_expertise=("domain_expertise", "mean"),
).reset_index()

governance_review = pd.DataFrame([
    {
        "area": "causal_question",
        "question": "Is the intervention question clearly stated?",
        "status": "complete",
        "owner": "Research / Product",
    },
    {
        "area": "estimand",
        "question": "Are treatment, outcome, unit, population, and time horizon documented?",
        "status": "partial",
        "owner": "Causal Analysis",
    },
    {
        "area": "interference",
        "question": "Could one unit's treatment affect another unit's outcome?",
        "status": "partial",
        "owner": "Experimentation",
    },
    {
        "area": "transportability",
        "question": "Can the estimated effect generalize to the target deployment context?",
        "status": "missing",
        "owner": "Governance",
    },
])

segment_effects, governance_review

## Interpretation

Causal machine learning should pair heterogeneous-effect estimation with governance documentation about assumptions, interference, and transportability.